# Notebook 12 — Spike detection + anchor check

**Spike rule (only):** per-topic z-score on growth-velocity **> 2.5** — each topic uses its own mean and spread so we flag unusually sharp *relative* jumps, not one global cutoff.

**Inputs:** `reports/topic_modeling/topics_over_time.csv` (from notebook 11), `data/df_clean.pkl`, `models/sbert_topic_bundle.joblib`.

**Outputs:** `reports/topic_modeling/spike_events.csv` (`topic_id`, `spike_date`, `velocity`, `velocity_z`, `keywords`), optional anchor CSVs. (Spikes are **not** recomputed in notebook 11 — only here.)


In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from IPython.display import display
import pandas as pd


In [2]:
from spike_events_from_topics_over_time import build_spike_table

OUT_DIR = os.path.join(PROJECT_ROOT, "reports", "topic_modeling")
spike_path = os.path.join(OUT_DIR, "spike_events.csv")
Z_THRESHOLD = 2.5
print(f"Spikes: z_threshold={Z_THRESHOLD}")
spike_df = build_spike_table(
    os.path.join(OUT_DIR, "topics_over_time.csv"),
    df_clean_path=os.path.join(PROJECT_ROOT, "data", "df_clean.pkl"),
    bundle_path=os.path.join(PROJECT_ROOT, "models", "sbert_topic_bundle.joblib"),
    z_threshold=Z_THRESHOLD,
)
spike_df.to_csv(spike_path, index=False)
print(f"Saved {len(spike_df)} spikes → {spike_path}")
display(spike_df.head(15))


Spikes: z_threshold=2.5


Saved 509 spikes → /Users/kk/ai_dl_project/dynamic-trend-event-detector/reports/topic_modeling/spike_events.csv


,topic_id,spike_date,velocity,velocity_z,keywords
0,61,2021-11-26,0.028224,6.162136,"omicron, variant, covid, says, nsw"
1,79,2021-11-26,0.006926,5.953068,"cleo, smith, police, audio, finding"
2,254,2021-09-19,0.014847,5.706029,"september, october"
3,247,2020-03-11,0.012500,5.636405,"coronavirus, afl, season, nrl, league"
4,203,2020-08-01,0.010305,5.200760,"beirut, explosion, blast, massive, lebanese"
5,246,2020-03-11,0.004310,5.153107,"sport, outbreak, events, amid, italian"
6,233,2020-03-11,0.008171,5.148751,"scott, economic, recession, stimulus, addresses"
7,38,2020-09-26,0.006818,5.102477,"azerbaijan, armenia, karabakh, nagorno, fighting"
8,586,2021-07-13,0.010867,5.097497,"nsw, new, records, local, recorded"
9,443,2020-03-11,0.007759,5.087620,"coronavirus, package, government, economic, fe..."


## Optional — ground-truth anchor table

Scores `spike_events.csv` against three annotated date windows (bushfires / COVID / lockdown).


In [3]:
from anchor_ground_truth_report import run_report

summary, detail = run_report(os.path.join(PROJECT_ROOT, "reports", "topic_modeling"))
display(summary)


,anchor_id,anchor_label,date_window,expected_keywords_probe,gdelt_theme_expected,detected,best_source,best_date,best_topic_id,best_keywords,probe_hits,note
0,black_summer,Black Summer bushfires,2019-11-01 .. 2020-01-31,"fire, smoke, evacuation, blaze, nsw, bushfire,...",ENV_FIRES,yes,spike_events,2019-12-23,139,"bushfire, bushfires, nsw, fires, victoria",4,GDELT verification pending — keyword overlap only
1,covid_first_au,First Australian COVID case,2020-01-20 .. 2020-02-05,"virus, coronavirus, china, health, travel, covid",HEALTH_PANDEMIC,yes,spike_events,2020-01-22,583,"coronavirus, china, wuhan, chinese, toll",3,GDELT verification pending — keyword overlap only
2,national_lockdown,National lockdown,2020-03-01 .. 2020-04-15,"lockdown, restrictions, border, quarantine",HEALTH_PANDEMIC,yes,spike_events,2020-03-11,581,"coronavirus, italy, lockdown, death, toll",1,GDELT verification pending — keyword overlap only
